# WAI-illustrious SDXL — tạo ảnh anime trên Google Colab

Notebook chạy **checkpoint SDXL đầy đủ** (`.safetensors`) bằng Diffusers, không cần WebUI/link chia sẻ công khai. Nếu chưa có model, ô 4 **tự tải WAI-illustrious v17** từ [bản lưu trên Hugging Face](https://huggingface.co/LyliaEngine/waiIllustriousSDXL_v170/blob/32be7bfdcd406db70df663b9cee3313957deb68f/waiIllustriousSDXL_v170.safetensors). File ~6,94 GB; SHA-256 được ghim và đối chiếu với [bản gốc Civitai](https://civitai.com/api/v1/model-versions/2883731). Không cần token, không tải thêm trọng số SDXL base.

**Tài nguyên tay/chân/mắt:** ô 5 có thể tự tải hai LoRA **Illustrious** đã đối chiếu SHA-256 với file phát hành trên Civitai: [Anatomy Helper V1](https://civitai.com/api/v1/model-versions/1318504) (tay, bàn chân, tư thế) và [Eyes for Illustrious V1](https://civitai.com/api/v1/model-versions/2066663) (mắt). Hai bản lưu HF được ghim commit, xác minh kích thước + SHA-256 **trước khi nạp**; mỗi file ~228 MB. Có thể tắt từng LoRA trong ô 3. Ô 8 cho phép **chọn vùng lỗi bằng hộp tọa độ hoặc ảnh mask trắng/đen để sửa bằng inpainting**, dùng lại checkpoint đang nạp, không tải thêm checkpoint 7 GB. Xác thực file **không** bảo đảm LoRA sửa được mọi lỗi; ảnh vẫn cần người dùng kiểm tra.

**Bắt đầu:**
1. Vào **Runtime → Change runtime type → GPU** (T4 hoặc GPU mạnh hơn). Chạy lần lượt các ô **1 → 7**, cho phép gắn Drive; chưa có checkpoint? `AUTO_DOWNLOAD=True` sẽ tải bản v17 và lưu vào Drive nếu có thể. **Đã có checkpoint riêng?** Sửa `MODEL_PATH` ở ô 3; file có sẵn được ưu tiên, không ghi đè và không được tự nhận là v17 nếu chưa đối chiếu hash.
2. Chọn/tắt LoRA và đặt cường độ trong ô 3 (mặc định bật cả hai). Sửa prompt ở ô 7 rồi chạy lại ô 7 để tạo ảnh mới. PNG mặc định lưu vào `MyDrive/AI/outputs`.
3. **Tùy chọn**, nếu ảnh vẫn lỗi tay/chân/mắt, mở ô 8, nhập vùng cần sửa (một ô mỗi lần thường cho kết quả dễ kiểm soát hơn). Vùng **trắng** của mask được thay đổi; vùng đen giữ nguyên. Không tự nhận diện vị trí tay/chân/mắt.

Cần khoảng 7–9 GiB đĩa trống để tải checkpoint và thêm tối đa ~457 MB cho hai LoRA; khi `/content` ít chỗ notebook thử tải trực tiếp vào Drive. Cấu hình/tokenizer SDXL được Diffusers lấy khi nạp lần đầu. RAM/VRAM và hạn mức phụ thuộc Colab; notebook không thể tự cấp GPU/RAM.

In [ ]:
# @title 1. Kiểm tra GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime → Change runtime type → GPU, rồi chạy lại ô này.")
device = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"GPU: {device.name} | VRAM trống: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# @title 2. Tự cài thư viện còn thiếu (giữ PyTorch/CUDA của Colab)
%pip -q install "diffusers==0.35.2" "transformers==4.52.4" "accelerate==1.10.1" "peft==0.17.1" "safetensors>=0.4.5,<1" "huggingface-hub==0.36.2" "hf-xet>=1.1.3,<2"

In [ ]:
# @title 3. Cấu hình model, LoRA, Drive và bộ nhớ { display-mode: "form" }
MOUNT_DRIVE = True # @param {type:"boolean"}
MODEL_PATH = "/content/drive/MyDrive/AI/models/WAI-illustrious.safetensors" # @param {type:"string"}
AUTO_DOWNLOAD = True # @param {type:"boolean"}
PERSIST_MODEL_TO_DRIVE = True # @param {type:"boolean"}
CACHE_MODEL_LOCAL = True # @param {type:"boolean"}
USE_ANATOMY_LORA = True # @param {type:"boolean"}
ANATOMY_WEIGHT = 0.55 # @param {type:"slider", min:0.1, max:1.0, step:0.05}
ANATOMY_LORA_PATH = "" # @param {type:"string"}
USE_EYE_LORA = True # @param {type:"boolean"}
EYE_WEIGHT = 0.45 # @param {type:"slider", min:0.1, max:1.0, step:0.05}
EYE_LORA_PATH = "" # @param {type:"string"}
PERSIST_LORAS_TO_DRIVE = True # @param {type:"boolean"}
OUTPUT_DIR = "/content/drive/MyDrive/AI/outputs" # @param {type:"string"}
VRAM_MODE = "auto" # @param ["auto", "speed", "low_vram"]

from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

source_model = Path(MODEL_PATH).expanduser()
output_dir = Path(OUTPUT_DIR).expanduser()
local_cache_root = Path("/content/wai_model_cache")
local_lora_cache = Path("/content/wai_lora_cache")
drive_root = Path("/content/drive")
lora_drive_dir = drive_root / "MyDrive" / "AI" / "loras"
if not source_model.is_absolute() or not output_dir.is_absolute():
    raise ValueError("MODEL_PATH và OUTPUT_DIR phải là đường dẫn tuyệt đối.")
if source_model.suffix.lower() != ".safetensors":
    raise ValueError("MODEL_PATH phải kết thúc bằng .safetensors (checkpoint đầy đủ, không phải LoRA).")
using_drive = any(drive_root == p or drive_root in p.parents for p in (source_model, output_dir))
if using_drive and not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive chưa được gắn. Bật MOUNT_DRIVE rồi chạy lại ô 3, hoặc đổi đường dẫn sang /content.")
if VRAM_MODE not in ("auto", "speed", "low_vram"):
    raise ValueError("VRAM_MODE phải là auto, speed hoặc low_vram.")
for weight in (ANATOMY_WEIGHT, EYE_WEIGHT):
    if not 0.1 <= weight <= 1.0:
        raise ValueError("Cường độ LoRA phải trong khoảng 0.1–1.0.")
print("Model có sẵn:" if source_model.is_file() else "Model chưa có (sẽ tự tải ở ô 4):", source_model)
print("LoRA bật:", ", ".join(name for enabled, name in ((USE_ANATOMY_LORA, "anatomy"), (USE_EYE_LORA, "eyes")) if enabled) or "không")
print("Thư mục lưu ảnh:", output_dir)

In [ ]:
# @title 4. Tự tải/checkpoint: ưu tiên file có sẵn, kiểm tra SHA-256 khi tải mới
import hashlib
import os
import shutil
# Tăng thời gian chờ trên mạng Colab chập chờn; đặt trước khi import huggingface_hub.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
from huggingface_hub import hf_hub_download
from safetensors import safe_open

# Bản v17 pruned FP16; hash trùng file Civitai model version 2883731.
HF_REPO = "LyliaEngine/waiIllustriousSDXL_v170"
HF_FILENAME = "waiIllustriousSDXL_v170.safetensors"
HF_REVISION = "32be7bfdcd406db70df663b9cee3313957deb68f"
HF_SHA256 = "f116b0c78ff441467b0cdc8f1936e1ed18ea31e9997c7b132b1b8db533f0bd04"
HF_MODEL_BYTES = 6_938_040_682
MIN_CHECKPOINT_BYTES = 100 * 2**20
DISK_RESERVE_BYTES = 2 * 2**30

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 2**20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def inspect_checkpoint(path, verify_official=False):
    path = Path(path)
    if path.suffix.lower() != ".safetensors" or not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy checkpoint .safetensors: {path}")
    size = path.stat().st_size
    if size < MIN_CHECKPOINT_BYTES:
        raise ValueError("File model quá nhỏ: có thể tải dở, là HTML hoặc là LoRA.")
    if verify_official:
        if size != HF_MODEL_BYTES or sha256_file(path) != HF_SHA256:
            raise ValueError("SHA-256/dung lượng model tải về không trùng bản v17 gốc; không nạp file này.")
    try:
        with safe_open(str(path), framework="pt", device="cpu") as header:
            keys = header.keys()  # chỉ đọc header, không tải trọng số vào RAM
            has_unet = any(key.startswith("model.diffusion_model.") for key in keys)
            has_clip = any(key.startswith("conditioner.embedders.") for key in keys)
    except Exception as exc:
        raise ValueError("Không đọc được safetensors; hãy kiểm tra file checkpoint.") from exc
    if not (has_unet and has_clip):
        raise ValueError("Cần checkpoint SDXL đầy đủ (UNet + text encoder), không phải LoRA/UNet-only.")
    return size

def copy_atomic(source, destination, size, preserve_mtime=False):
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".partial")
    try:
        # Drive FUSE có thể không cho phép copy2/copystat; chỉ giữ mtime khi cache vào /content.
        if preserve_mtime:
            shutil.copy2(source, partial)
        else:
            shutil.copyfile(source, partial)
        if partial.stat().st_size != size:
            raise OSError("Bản sao checkpoint không đầy đủ.")
        os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)

if source_model.exists() and not source_model.is_file():
    raise ValueError(f"MODEL_PATH là thư mục hoặc tệp đặc biệt, không phải checkpoint: {source_model}")
if source_model.is_file():
    model_size = inspect_checkpoint(source_model)  # file của người dùng có thể là phiên bản khác
    checkpoint = source_model
    if CACHE_MODEL_LOCAL and drive_root in source_model.parents:
        local_cache_root.mkdir(parents=True, exist_ok=True)
        cached = local_cache_root / source_model.name
        same_file = (
            cached.is_file()
            and cached.stat().st_size == model_size
            and cached.stat().st_mtime_ns == source_model.stat().st_mtime_ns
        )
        if same_file:
            checkpoint = cached
            print("Dùng lại bản sao /content trong phiên này.")
        elif shutil.disk_usage(local_cache_root).free >= model_size + DISK_RESERVE_BYTES:
            try:
                print("Sao chép model từ Drive sang /content để nạp nhanh hơn...")
                copy_atomic(source_model, cached, model_size, preserve_mtime=True)
                checkpoint = cached
            except OSError as exc:
                print(f"Không sao chép được ({type(exc).__name__}); nạp trực tiếp từ Drive.")
        else:
            print("Đĩa /content không đủ chỗ để sao chép; nạp trực tiếp từ Drive.")
    # Băm đúng file SẼ nạp, kể cả bản sao/cache /content (có thể khác nguồn Drive).
    if model_size == HF_MODEL_BYTES:
        checkpoint_hash = sha256_file(checkpoint)
        if checkpoint_hash != HF_SHA256 and checkpoint != source_model:
            if sha256_file(source_model) == HF_SHA256:
                print("Cache /content sai SHA-256; bỏ qua bản sao và nạp bản gốc trên Drive.")
                checkpoint = source_model
                checkpoint_hash = HF_SHA256
        if checkpoint_hash == HF_SHA256:
            print("Checkpoint có sẵn: đã xác minh là WAI-illustrious v17 (SHA-256 trùng Civitai).")
        else:
            print("Checkpoint có sẵn: chỉ kiểm tra định dạng SDXL, KHÔNG xác thực là WAI v17; LoRA chỉ hợp với họ Illustrious.")
    else:
        print("Checkpoint có sẵn: chỉ kiểm tra định dạng SDXL, KHÔNG xác thực là WAI v17; LoRA chỉ hợp với họ Illustrious.")
else:
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f"Chưa có model: {source_model}. Bật AUTO_DOWNLOAD ở ô 3 hoặc đặt checkpoint vào đúng đường dẫn.")
    local_cache_root.mkdir(parents=True, exist_ok=True)
    disk_ok = shutil.disk_usage(local_cache_root).free >= HF_MODEL_BYTES + DISK_RESERVE_BYTES
    on_drive = drive_root in source_model.parents
    print(f"Chưa có checkpoint; đang tải bản WAI-illustrious v17 ({HF_MODEL_BYTES / 10**9:.2f} GB)...")
    if disk_ok:
        try:
            downloaded = Path(hf_hub_download(
                repo_id=HF_REPO, filename=HF_FILENAME, revision=HF_REVISION,
                cache_dir=str(local_cache_root), token=False,
            ))
        except Exception as exc:
            raise RuntimeError("Không tải được model từ Hugging Face. Kiểm tra Internet hoặc tự đặt file ở MODEL_PATH.") from exc
        inspect_checkpoint(downloaded, verify_official=True)
        checkpoint = downloaded  # không sao chép lại sang /content lần thứ hai
        if on_drive and PERSIST_MODEL_TO_DRIVE:
            try:
                print("Lưu bản đã xác minh vào Drive để phiên sau không tải lại...")
                copy_atomic(downloaded, source_model, HF_MODEL_BYTES)
            except OSError as exc:
                print(f"Drive không lưu được ({type(exc).__name__}); vẫn tạo ảnh từ bản tạm ở /content.")
    elif on_drive and PERSIST_MODEL_TO_DRIVE:
        # Đĩa máy Colab quá ít: tải vào Drive, không tạo bản sao 7 GB trên /content.
        source_model.parent.mkdir(parents=True, exist_ok=True)
        try:
            downloaded = Path(hf_hub_download(
                repo_id=HF_REPO, filename=HF_FILENAME, revision=HF_REVISION,
                local_dir=str(source_model.parent), token=False,
            ))
            inspect_checkpoint(downloaded, verify_official=True)
            if downloaded != source_model:
                os.replace(downloaded, source_model)
            checkpoint = source_model
        except Exception as exc:
            raise RuntimeError("Không đủ đĩa /content và tải vào Drive không thành công. Kiểm tra dung lượng/quyền Drive.") from exc
    else:
        raise OSError("Thiếu đĩa trống để tải model (~9 GiB). Giải phóng đĩa, hoặc bật lưu vào Drive ở ô 3.")
    print("Đã đối chiếu SHA-256 với bản v17 gốc.")
print(f"Sẵn sàng: {checkpoint} ({checkpoint.stat().st_size / 2**30:.2f} GiB)")

In [ ]:
# @title 5. Tự tải và xác minh LoRA Illustrious cho tay/chân/mắt (có thể tắt ở ô 3)
# HF là bản lưu của bên thứ ba; SHA-256 đầy đủ trùng file phát hành Civitai ở các version ID sau.
ANATOMY_LORA_REPO = "ench100/bodyandface"
ANATOMY_LORA_FILE = "anatomy_helper.safetensors"
ANATOMY_LORA_REV = "bed49d45df95c0695aedad3b2aa6aff389fb3777"
ANATOMY_LORA_SHA256 = "bf6a950036b7599212a2c68d65f3ba07b28689067e167915d2a0ecb2018c26ca"
ANATOMY_LORA_BYTES = 228_473_940  # Civitai version 1318504, file 1223247
EYE_LORA_REPO = "Muapi/eyes-for-illustrious-lora-perfect-anime-eyes"
EYE_LORA_FILE = "eyes-for-illustrious-lora-perfect-anime-eyes.safetensors"
EYE_LORA_REV = "1abbc862f53f5101962ebf1c337513aff91bd206"
EYE_LORA_SHA256 = "97c1a083ffe6b4d45c545196eabd01c754936b996ade0c9db6d072f3bd340c55"
EYE_LORA_BYTES = 228_457_660  # Civitai version 2066663, file 1963176
LORA_DISK_RESERVE_BYTES = 256 * 2**20
if "checkpoint" not in globals():
    raise RuntimeError("Hãy chạy ô 4 để chuẩn bị checkpoint trước khi tải LoRA.")

lora_manifest = {
    "anatomy": dict(enabled=USE_ANATOMY_LORA, weight=ANATOMY_WEIGHT, manual=ANATOMY_LORA_PATH,
                    repo=ANATOMY_LORA_REPO, filename=ANATOMY_LORA_FILE, revision=ANATOMY_LORA_REV,
                    sha256=ANATOMY_LORA_SHA256, size=ANATOMY_LORA_BYTES, version="Civitai:1318504"),
    "eyes": dict(enabled=USE_EYE_LORA, weight=EYE_WEIGHT, manual=EYE_LORA_PATH,
                 repo=EYE_LORA_REPO, filename=EYE_LORA_FILE, revision=EYE_LORA_REV,
                 sha256=EYE_LORA_SHA256, size=EYE_LORA_BYTES, version="Civitai:2066663"),
}

def inspect_lora(path, spec):
    path = Path(path)
    if path.suffix.lower() != ".safetensors" or not path.is_file():
        raise FileNotFoundError(f"LoRA cần là file .safetensors: {path}")
    if path.stat().st_size != spec["size"] or sha256_file(path) != spec["sha256"]:
        raise ValueError(f"LoRA {spec['version']} không đúng kích thước/SHA-256: {path}. Không nạp file sai phiên bản.")
    try:
        with safe_open(str(path), framework="pt", device="cpu") as header:
            if not any(key.startswith("lora_unet_") or key.startswith("unet.") for key in header.keys()):
                raise ValueError("Không có lớp LoRA UNet kiểu SDXL/Kohya.")
    except Exception as exc:
        raise ValueError(f"Không đọc được header LoRA: {path}") from exc
    return path

def prepare_lora(spec):
    manual = spec["manual"].strip()
    if manual:
        return inspect_lora(Path(manual).expanduser(), spec)  # chấp nhận tên file Civitai nếu SHA trùng
    have_drive = (drive_root / "MyDrive").is_dir()
    saved = lora_drive_dir / spec["filename"]
    if have_drive and saved.is_file():
        print(f"Dùng lại bản LoRA đã xác minh trên Drive: {saved}")
        return inspect_lora(saved, spec)
    local_lora_cache.mkdir(parents=True, exist_ok=True)
    disk_ok = shutil.disk_usage(local_lora_cache).free >= spec["size"] + LORA_DISK_RESERVE_BYTES
    if not disk_ok and not (have_drive and PERSIST_LORAS_TO_DRIVE):
        raise OSError("Thiếu đĩa /content cho LoRA. Giải phóng ~0.5 GiB, hoặc gắn Drive và bật PERSIST_LORAS_TO_DRIVE.")
    print(f"Tải LoRA {spec['version']} từ Hugging Face, commit {spec['revision'][:8]}...")
    try:
        args = dict(repo_id=spec["repo"], filename=spec["filename"],
                    revision=spec["revision"], token=False)
        if disk_ok:
            args["cache_dir"] = str(local_lora_cache)  # không tạo thêm một bản trong /content
        else:
            lora_drive_dir.mkdir(parents=True, exist_ok=True)
            args["local_dir"] = str(lora_drive_dir)  # ít đĩa: tải thẳng vào Drive
        downloaded = inspect_lora(Path(hf_hub_download(**args)), spec)
    except Exception as exc:
        raise RuntimeError(f"LoRA {spec['version']} tải/xác minh thất bại. Dùng file gốc tại ANATOMY_LORA_PATH/EYE_LORA_PATH, hoặc tắt LoRA này ở ô 3.") from exc
    if disk_ok and have_drive and PERSIST_LORAS_TO_DRIVE:
        try:
            copy_atomic(downloaded, saved, spec["size"])
            try:
                inspect_lora(saved, spec)  # xác minh cả bản trên Drive, không chỉ kiểm tra size
            except ValueError:
                saved.unlink(missing_ok=True)  # chỉ xóa bản vừa do notebook sao chép
                raise
            print(f"Đã lưu bản sao xác minh trên Drive: {saved}")
        except (OSError, ValueError) as exc:
            print(f"Không lưu được LoRA lên Drive ({type(exc).__name__}); vẫn dùng bản đã xác minh trong /content.")
    return downloaded

lora_paths = {}
for name, spec in lora_manifest.items():
    if spec["enabled"]:
        lora_paths[name] = prepare_lora(spec)
        print(f"Đã xác minh {name}: {spec['version']} | SHA-256 {spec['sha256']}")
if not lora_paths:
    print("Không bật LoRA; ảnh vẫn dùng checkpoint gốc. Bật ở ô 3 rồi chạy lại ô 5–7 nếu cần.")

In [ ]:
# @title 6. Nạp model + LoRA, tự chọn GPU nhanh / CPU offload khi thiếu VRAM
import gc
import psutil
from diffusers import EulerAncestralDiscreteScheduler, StableDiffusionXLPipeline

requested_loras = {name for enabled, name in ((USE_ANATOMY_LORA, "anatomy"), (USE_EYE_LORA, "eyes")) if enabled}
if "lora_paths" not in globals():
    if requested_loras:
        raise RuntimeError("Hãy chạy ô 5 để tải/xác minh LoRA trước khi nạp model.")
elif set(lora_paths) != requested_loras or any(
    (lora_manifest[name]["weight"], lora_manifest[name]["manual"]) != (weight, manual)
    for name, weight, manual in
    (("anatomy", ANATOMY_WEIGHT, ANATOMY_LORA_PATH), ("eyes", EYE_WEIGHT, EYE_LORA_PATH))
    if name in requested_loras
):
    raise RuntimeError("Cấu hình LoRA đã đổi: chạy lại ô 5 để xác minh trước khi nạp model.")
if "pipe" in globals():
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
ram_gib = psutil.virtual_memory().available / 2**30
print(f"RAM hệ thống khả dụng: {ram_gib:.1f} GiB")
if ram_gib < 8:
    print("Cảnh báo: nạp checkpoint 6,94 GB có thể cần Colab high-RAM nếu phiên này quá ít RAM.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
free_bytes, total_bytes = torch.cuda.mem_get_info()
# Ưu tiên GPU trực tiếp trên T4 (~15 GiB) thay vì ép CPU offload gần như mọi lần.
# Chỉ chọn offload trước nếu VRAM thực sự thiếu; OOM khi nạp/tạo ảnh vẫn tự thử lại.
auto_min_vram = (12.5 + 0.4 * len(globals().get("lora_paths", {}))) * 2**30
print(f"VRAM trống trước khi nạp: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB; ngưỡng chạy GPU trực tiếp: {auto_min_vram / 2**30:.1f} GiB")
use_offload = VRAM_MODE == "low_vram" or (VRAM_MODE == "auto" and free_bytes < auto_min_vram)

def create_pipeline(offload):
    pipeline = StableDiffusionXLPipeline.from_single_file(
        str(checkpoint), torch_dtype=torch.float16, use_safetensors=True,
    )
    pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(pipeline.scheduler.config)
    pipeline.vae.enable_slicing()
    pipeline.vae.enable_tiling()  # giảm đỉnh VRAM khi giải mã ảnh lớn, cả chế độ GPU trực tiếp
    names = []
    weights = []
    for name, spec in globals().get("lora_manifest", {}).items():
        if name in lora_paths:
            path = inspect_lora(lora_paths[name], spec)  # kiểm lại ngay trước mỗi lần nạp / nạp lại khi OOM
            pipeline.load_lora_weights(
                str(path.parent), weight_name=path.name, adapter_name=name,
                local_files_only=True, use_safetensors=True,
            )
            names.append(name)
            weights.append(spec["weight"])
    if names:
        pipeline.set_adapters(names, adapter_weights=weights)
        print("LoRA đang dùng:", ", ".join(f"{name}={weight:.2f}" for name, weight in zip(names, weights)))
    if offload:
        pipeline.enable_model_cpu_offload()
    else:
        pipeline.to("cuda")
    return pipeline

print("Chế độ:", "CPU offload (vẫn dùng GPU từng phần, tốn RAM hệ thống)" if use_offload else "GPU trực tiếp (FP16)")
if VRAM_MODE == "low_vram":
    print("VRAM_MODE=low_vram đang ép CPU offload; chọn auto ở ô 3 nếu muốn ưu tiên GPU.")
gpu_oom = False
try:
    pipe = create_pipeline(use_offload)
except torch.cuda.OutOfMemoryError:
    gpu_oom = True
if gpu_oom:
    if VRAM_MODE != "auto" or use_offload:
        raise RuntimeError("Hết VRAM khi nạp model/LoRA. Tắt bớt LoRA hoặc chọn VRAM_MODE='low_vram', rồi chạy lại ô 5–6.")
    gc.collect()
    torch.cuda.empty_cache()
    print("Không đủ VRAM để nạp trực tiếp; đang tự thử CPU offload...")
    pipe = create_pipeline(True)
    use_offload = True
print("Chế độ sau khi nạp:", "CPU offload (GPU tính toán từng phần)" if use_offload else "GPU trực tiếp")
free_after_load, _ = torch.cuda.mem_get_info()
print(f"VRAM trống sau khi nạp: {free_after_load / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
print("Model đã sẵn sàng. Chạy ô 7 để tạo ảnh; ô 8 sửa vùng lỗi nếu cần.")

In [ ]:
# @title 7. Tạo ảnh (sửa prompt/seed rồi chạy lại tùy thích) { display-mode: "form" }
PROMPT = "general, 1girl, solo, cherry blossoms, spring, soft sunlight, detailed eyes, anime illustration, masterpiece, best quality" # @param {type:"string"}
NEGATIVE_PROMPT = "nsfw, explicit, lowres, worst quality, bad anatomy, blurry" # @param {type:"string"}
SIZE = "1024x1024" # @param ["1024x1024", "832x1216", "1216x832", "768x1024", "1024x768", "1024x1344", "1344x1024"]
STEPS = 25 # @param {type:"slider", min:10, max:45, step:1}
CFG = 6.0 # @param {type:"slider", min:1, max:12, step:0.5}
SEED = -1 # @param {type:"integer"}
EMBED_METADATA = True # @param {type:"boolean"}

import json
import secrets
from datetime import datetime, timezone
from IPython.display import display
from PIL.PngImagePlugin import PngInfo

if "pipe" not in globals():
    raise RuntimeError("Chưa nạp model. Hãy chạy ô 5 trước.")
if not PROMPT.strip():
    raise ValueError("PROMPT không được để trống.")
width, height = map(int, SIZE.split("x"))
if min(width, height) < 512 or width % 8 or height % 8:
    raise ValueError("Kích thước ảnh phải >= 512 và chia hết cho 8.")
if not 1 <= STEPS <= 50 or not 1 <= CFG <= 12:
    raise ValueError("STEPS phải từ 1–50 và CFG từ 1–12.")
if SEED != -1 and not 0 <= SEED < 2**32:
    raise ValueError("SEED phải là -1 (ngẫu nhiên) hoặc số nguyên từ 0 đến 2^32 - 1.")
seed = secrets.randbelow(2**32) if SEED == -1 else SEED
positive_prompt = PROMPT.strip()
if "eyes" in globals().get("lora_paths", {}) and "perfect eyes" not in positive_prompt.lower():
    positive_prompt += ", perfect eyes"  # trigger tác giả công bố cho Eyes for Illustrious V1

def run_generation():
    generator = torch.Generator(device="cpu").manual_seed(seed)
    with torch.inference_mode():
        return pipe(
            prompt=positive_prompt, negative_prompt=NEGATIVE_PROMPT.strip(),
            width=width, height=height, num_inference_steps=STEPS,
            guidance_scale=CFG, num_images_per_prompt=1, generator=generator,
        ).images[0]

gpu_oom = False
try:
    image = run_generation()
except torch.cuda.OutOfMemoryError:
    gpu_oom = True
if gpu_oom:
    torch.cuda.empty_cache()
    if VRAM_MODE == "auto" and not use_offload:
        print("Hết VRAM khi tạo ảnh; đang tự chuyển sang CPU offload và thử lại cùng seed...")
        del pipe
        gc.collect()
        torch.cuda.empty_cache()
        pipe = create_pipeline(True)
        use_offload = True
        try:
            image = run_generation()
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            raise RuntimeError("Vẫn thiếu VRAM. Chọn kích thước 768x1024 hoặc 1024x768 rồi chạy lại ô 6.") from exc
    else:
        raise RuntimeError("Hết VRAM. Giảm kích thước, hoặc đặt VRAM_MODE='low_vram' và chạy lại ô 5–6.")

display(image)
png_info = PngInfo()
if EMBED_METADATA:
    png_info.add_text("parameters", json.dumps({
        "model": checkpoint.name,
        "prompt": positive_prompt, "negative_prompt": NEGATIVE_PROMPT.strip(),
        "loras": [{"name": name, "weight": spec["weight"], "version": spec["version"],
                   "sha256": spec["sha256"]} for name, spec in globals().get("lora_manifest", {}).items()
                  if name in globals().get("lora_paths", {})],
        "seed": seed, "width": width, "height": height,
        "steps": STEPS, "cfg": CFG, "sampler": "Euler a",
    }, ensure_ascii=False))
filename = f"wai_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}_{seed}.png"

def save_image(directory):
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / filename
    partial = directory / (filename + ".partial")
    try:
        image.save(partial, format="PNG", pnginfo=png_info)
        os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    return destination

try:
    if (output_dir == drive_root or drive_root in output_dir.parents) and not Path("/content/drive/MyDrive").is_dir():
        raise OSError("Drive đã ngắt kết nối")
    output_path = save_image(output_dir)
except OSError as exc:
    backup_dir = Path("/content/wai_outputs")
    if output_dir == backup_dir:
        raise
    print(f"Không lưu được vào {output_dir} ({type(exc).__name__}); lưu tạm ở {backup_dir}.")
    output_path = save_image(backup_dir)
print(f"Seed: {seed} | Đã lưu: {output_path}")

In [ ]:
# @title 8. Tùy chọn: sửa tay/chân/mắt bằng inpainting vùng được chọn { display-mode: "form" }
SOURCE_IMAGE = "" # @param {type:"string"}
MASK_PATH = "" # @param {type:"string"}
BOXES = "" # @param {type:"string"}
TARGET = "hands" # @param ["hands", "legs", "eyes", "custom"]
REFINE_PROMPT = "" # @param {type:"string"}
REFINE_STRENGTH = 0.45 # @param {type:"slider", min:0.2, max:0.85, step:0.05}
REFINE_STEPS = 25 # @param {type:"slider", min:10, max:45, step:1}
REFINE_SEED = -1 # @param {type:"integer"}
FEATHER_PX = 8 # @param {type:"slider", min:0, max:24, step:2}

# BOXES ví dụ: 100,300,240,490;700,400,850,590 (x1,y1,x2,y2; tọa độ phải nằm trong ảnh).
# Nếu dùng MASK_PATH: màu trắng = sửa, đen = giữ. Để trống SOURCE_IMAGE để sửa ảnh vừa tạo ở ô 7.
import json
import os
import secrets
from datetime import datetime, timezone
from IPython.display import display
from PIL.PngImagePlugin import PngInfo
from PIL import Image, ImageChops, ImageDraw, ImageFilter, ImageOps
from diffusers import AutoPipelineForInpainting

if "pipe" not in globals():
    raise RuntimeError("Chưa nạp model. Chạy ô 1–6 trước.")
source_path = Path(SOURCE_IMAGE.strip()).expanduser() if SOURCE_IMAGE.strip() else globals().get("output_path")
if not source_path or not Path(source_path).is_file():
    raise FileNotFoundError("Chưa có ảnh nguồn; chạy ô 7 trước hoặc đặt SOURCE_IMAGE tới ảnh PNG/JPG trên Drive.")
if bool(MASK_PATH.strip()) == bool(BOXES.strip()):
    raise ValueError("Chọn đúng MỘT cách tạo mask: MASK_PATH (trắng/đen) HOẶC BOXES (tọa độ x1,y1,x2,y2;...).")
if not 0.2 <= REFINE_STRENGTH <= 0.85 or not 10 <= REFINE_STEPS <= 45:
    raise ValueError("REFINE_STRENGTH từ 0.2–0.85 và REFINE_STEPS từ 10–45.")
if REFINE_SEED != -1 and not 0 <= REFINE_SEED < 2**32:
    raise ValueError("REFINE_SEED phải là -1 hoặc từ 0 đến 2^32 - 1.")
with Image.open(source_path) as raw:
    original = ImageOps.exif_transpose(raw).convert("RGB")
w, h = original.size
if min(w, h) < 512 or w % 8 or h % 8:
    raise ValueError("Ảnh phải có cạnh >= 512 và kích thước chia hết cho 8; hãy resize ảnh trước khi sửa.")
if MASK_PATH.strip():
    with Image.open(Path(MASK_PATH.strip()).expanduser()) as raw_mask:
        mask = raw_mask.convert("L")
    if mask.size != original.size:
        raise ValueError("Kích thước mask phải trùng ảnh nguồn. Trắng = sửa; đen = giữ nguyên.")
else:
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)
    boxes = BOXES.split(";")
    if len(boxes) > 8:
        raise ValueError("Tối đa 8 hộp; sửa từng vùng nhỏ giúp kiểm soát kết quả tốt hơn.")
    for text in boxes:
        try:
            x1, y1, x2, y2 = (int(value.strip()) for value in text.split(","))
        except ValueError as exc:
            raise ValueError("BOXES phải có dạng: x1,y1,x2,y2; x1,y1,x2,y2") from exc
        if not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h):
            raise ValueError(f"Hộp {text!r} nằm ngoài ảnh {w}x{h} hoặc không có diện tích.")
        draw.rectangle((x1, y1, x2 - 1, y2 - 1), fill=255)
# Ngưỡng nhị phân cho Diffusers; feather khi ghép lại để bớt lộ viền.
mask_binary = mask.point(lambda value: 255 if value >= 128 else 0)
if mask_binary.getbbox() is None or mask_binary.getextrema() == (255, 255):
    raise ValueError("Vùng trắng phải có diện tích > 0 nhưng không phủ toàn bộ ảnh.")
print(f"Ảnh: {w}x{h}. Vùng trắng sẽ được vẽ lại ({TARGET}).")
display(mask_binary)

hints = {
    "hands": ("natural hands, correct number of fingers, detailed fingers", "extra fingers, missing fingers, fused fingers, deformed hands"),
    "legs": ("natural leg anatomy, well-formed feet, balanced pose", "extra legs, broken legs, deformed feet, extra toes"),
    "eyes": ("symmetrical eyes, detailed irises, perfect eyes", "misaligned eyes, deformed eyes, extra eyes"),
    "custom": ("", ""),
}
base_prompt = REFINE_PROMPT.strip() or globals().get("PROMPT", "").strip()
if not base_prompt:
    raise ValueError("Nhập REFINE_PROMPT khi sửa ảnh không được tạo từ ô 7.")
hint_pos, hint_neg = hints[TARGET]
repair_prompt = ", ".join(part for part in (base_prompt, hint_pos) if part)
repair_negative = ", ".join(part for part in (globals().get("NEGATIVE_PROMPT", ""), hint_neg) if part)
if "eyes" in globals().get("lora_paths", {}) and "perfect eyes" not in repair_prompt.lower():
    repair_prompt += ", perfect eyes"
repair_seed = secrets.randbelow(2**32) if REFINE_SEED == -1 else REFINE_SEED

def inpaint_once():
    # from_pipe chia sẻ UNet/VAE/text encoder; không tải thêm checkpoint hay thêm bản 7 GB trong RAM.
    repair_pipe = AutoPipelineForInpainting.from_pipe(pipe)
    if use_offload:
        # Chuyển chủ sở hữu hook CPU offload sang pipeline inpaint rồi trả lại pipeline tạo ảnh.
        pipe.remove_all_hooks()
        try:
            repair_pipe.enable_model_cpu_offload()
        except Exception:
            pipe.enable_model_cpu_offload()
            raise
    try:
        generator = torch.Generator(device="cpu").manual_seed(repair_seed)
        with torch.inference_mode():
            return repair_pipe(
                prompt=repair_prompt, negative_prompt=repair_negative,
                image=original, mask_image=mask_binary, width=w, height=h,
                strength=REFINE_STRENGTH, num_inference_steps=REFINE_STEPS,
                guidance_scale=globals().get("CFG", 6.0), padding_mask_crop=32, generator=generator,
            ).images[0]
    finally:
        if use_offload:
            repair_pipe.remove_all_hooks()
            pipe.enable_model_cpu_offload()

inpaint_oom = False
try:
    inpaint_result = inpaint_once()
except torch.cuda.OutOfMemoryError:
    inpaint_oom = True
if inpaint_oom:
    torch.cuda.empty_cache()
    if VRAM_MODE != "auto" or use_offload:
        raise RuntimeError("Thiếu VRAM khi sửa vùng ảnh. Dùng ảnh 768x1024 hoặc chọn VRAM_MODE='low_vram' rồi chạy lại ô 6–8.")
    # Ra khỏi khối except trước khi nạp lại: giải phóng traceback đang giữ pipeline GPU cũ.
    print("Hết VRAM khi sửa; nạp lại bằng CPU offload và thử cùng seed...")
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    pipe = create_pipeline(True)  # sử dụng lại file checkpoint + LoRA đã xác minh, không tải mới
    use_offload = True
    try:
        inpaint_result = inpaint_once()
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise RuntimeError("Vẫn thiếu VRAM khi sửa. Giảm kích thước ảnh hoặc tắt bớt LoRA ở ô 3.") from exc
if inpaint_result.size != original.size:
    raise ValueError("Pipeline trả về ảnh sai kích thước; không ghép ảnh để tránh lệch mask.")
# Làm mềm phía TRONG vùng trắng; chặn tràn ra vùng đen để nền ngoài mask nguyên vẹn.
blend_mask = ImageChops.multiply(mask_binary, mask_binary.filter(ImageFilter.GaussianBlur(radius=FEATHER_PX))) if FEATHER_PX else mask_binary
repaired = Image.composite(inpaint_result.convert("RGB"), original, blend_mask)
display(repaired)
repair_info = PngInfo()
if globals().get("EMBED_METADATA", True):
    repair_info.add_text("parameters", json.dumps({
        "operation": "masked_inpainting", "source": str(source_path), "target": TARGET,
        "model": checkpoint.name, "prompt": repair_prompt, "negative_prompt": repair_negative,
        "seed": repair_seed, "steps": REFINE_STEPS, "strength": REFINE_STRENGTH,
        "mask": MASK_PATH.strip() or BOXES.strip(),
        "loras": [{"name": name, "version": spec["version"], "weight": spec["weight"],
                   "sha256": spec["sha256"]} for name, spec in globals().get("lora_manifest", {}).items()
                  if name in globals().get("lora_paths", {})],
    }, ensure_ascii=False))
repair_filename = f"wai_repair_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}_{repair_seed}.png"

def save_repair(directory):
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / repair_filename
    partial = directory / (repair_filename + ".partial")
    try:
        repaired.save(partial, format="PNG", pnginfo=repair_info)
        os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    return destination

try:
    if (output_dir == drive_root or drive_root in output_dir.parents) and not (drive_root / "MyDrive").is_dir():
        raise OSError("Drive đã ngắt kết nối")
    repaired_path = save_repair(output_dir)
except OSError as exc:
    backup_dir = Path("/content/wai_outputs")
    if output_dir == backup_dir:
        raise
    print(f"Không lưu được vào {output_dir} ({type(exc).__name__}); lưu tạm ở {backup_dir}.")
    repaired_path = save_repair(backup_dir)
print(f"Seed sửa vùng: {repair_seed} | Đã lưu: {repaired_path}")

### Phiên bản tài nguyên, mẹo và xử lý sự cố

- **Đối chiếu phiên bản:** checkpoint WAI v17, Anatomy Helper V1 (Illustrious, Civitai `1318504`) và Eyes for Illustrious V1 (Illustrious, Civitai `2066663`) được ghim file/commit HF + kích thước + SHA-256 trùng metadata Civitai. Bản HF là **mirror bên thứ ba**, không phải tài khoản tác giả Civitai. Notebook kiểm tra **toàn bộ SHA-256** trước khi nạp, không chỉ mã AutoV2 ngắn. File do bạn tự cung cấp tại `MODEL_PATH` chỉ kiểm tra header SDXL; nếu đúng kích thước + hash v17 thì xác nhận v17, nếu không thì **chưa xác thực phiên bản**. File LoRA nhập bằng `ANATOMY_LORA_PATH`/`EYE_LORA_PATH` vẫn bắt buộc trùng hash bản ghim.
- **Tải/lưu tài nguyên:** nếu thiếu checkpoint/LoRA, bật cờ tải ở ô 3, chạy theo thứ tự ô 4–6. Model ~6,94 GB, mỗi LoRA ~228 MB; nếu Drive hết quota, vẫn dùng bản cache /content đã xác minh. Nếu /content quá ít chỗ, thử tải trực tiếp vào Drive. LoRA lưu ở `MyDrive/AI/loras` khi `PERSIST_LORAS_TO_DRIVE=True`. Có thể tắt từng LoRA để giảm đĩa/VRAM, và rerun ô 5–7 sau khi đổi cấu hình. File có sẵn sai hash sẽ **bị từ chối, không bị tự ghi đè**.
- **Sửa vùng lỗi ở ô 8:** xem ảnh ô 7 và nhập `BOXES=x1,y1,x2,y2` (tọa độ pixel; có thể cách nhau bằng `;`), hoặc vẽ ảnh mask cùng kích thước: **trắng sửa, đen giữ** rồi đặt `MASK_PATH`. Để `SOURCE_IMAGE` rỗng nếu sửa ảnh vừa tạo; có thể sửa PNG khác bằng đường dẫn. Chọn `TARGET=hands/legs/eyes`, thử strength 0.35–0.55, sửa mỗi vùng nhỏ một lượt; tăng strength sẽ làm lệch nhân vật/nét vẽ. Đây là inpainting dùng UNet SDXL 4 kênh thông thường (không tải checkpoint inpaint riêng), **không tự xác định tay/chân/mắt, không bảo đảm hết lỗi**. Ảnh gốc không bị ghi đè.
- **Thiếu đĩa/RAM/VRAM:** `CACHE_MODEL_LOCAL=True` chỉ sao chép checkpoint Drive khi đủ đĩa; `VRAM_MODE=auto` ưu tiên GPU trực tiếp trên T4 khi VRAM trống ≥ 12,5 GiB + 0,4 GiB/LoRA; dưới ngưỡng mới dùng CPU offload, hoặc tự thử lại bằng offload khi OOM. VAE tiling áp dụng ở cả hai chế độ để giảm đỉnh VRAM; `low_vram` ép offload (GPU vẫn tính toán từng phần, RAM hệ thống tăng, chậm hơn). Nếu vẫn OOM, dùng ảnh 768x1024/1024x768; RAM hệ thống thiếu thì cần runtime high-RAM. Notebook không thể tự cấp GPU/RAM.
- **Lưu ý chất lượng/nội dung:** LoRA anatomy được tác giả mô tả huấn luyện bằng dữ liệu hỗn hợp, có ảnh thiên về bàn chân; có thể ảnh hưởng phong cách/nội dung, hãy tắt nếu không phù hợp. Eye LoRA dùng từ khóa `perfect eyes`; tự thêm khi bật. Cả hai được đào tạo cho họ Illustrious SDXL, không dùng bản SD1.5/Pony/Anima thay thế. Xác thực byte/nguồn file **không** phải kiểm định chất lượng ảnh trên mọi prompt; so sánh seed, xem kỹ tay/chân/mắt, tuân thủ giấy phép và điều khoản model/Colab.
- `SEED=-1` chọn seed ngẫu nhiên; một ảnh mỗi lượt. PNG có thể chứa prompt/nguồn trong metadata nếu `EMBED_METADATA=True`; tắt trước khi chia sẻ nếu cần. Prompt mặc định lành mạnh nhưng không đảm bảo lọc nội dung; Drive ngắt khi lưu ảnh thì file dự phòng nằm ở `/content/wai_outputs` (hãy tải trước khi phiên hết).